In [ ]:
%pip install pandas
%pip install dash

In [ ]:
import pandas as pd

df = pd.read_csv('/Users/pawan/Downloads/E-Commerce Analysis/ecommerce_customer_churn_dataset.csv')
df.head()

In [ ]:
# Remove invalid ages
df = df[df["Age"].isna() | df["Age"].between(16, 100)]

# Fix values that should not be negative or above 100
df["Total_Purchases"] = df["Total_Purchases"].clip(lower=0)
df["Cart_Abandonment_Rate"] = df["Cart_Abandonment_Rate"].clip(0, 100)
df["Returns_Rate"] = df["Returns_Rate"].clip(0, 100)
df["Discount_Usage_Rate"] = df["Discount_Usage_Rate"].clip(0, 100)

# Fill missing values
df["Wishlist_Items"] = df["Wishlist_Items"].fillna(0)

# Fill missing numeric column values with median
numeric_cols = df.select_dtypes(include="number").columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())

# Convert selected columns to integers
int_cols = [
    "Age",
    "Customer_Service_Calls",
    "Product_Reviews_Written",
    "Wishlist_Items",
    "Payment_Method_Diversity",
    "Days_Since_Last_Purchase",
    "Login_Frequency"
]

df[int_cols] = df[int_cols].round().astype(int)

# Round remaining float columns to 2 decimal places
float_cols = df.select_dtypes(include="float").columns
df[float_cols] = df[float_cols].round(2)

# Clean text columns
text_cols = ["Gender", "Country", "City", "Signup_Quarter"]

for col in text_cols:
    df[col] = df[col].str.strip().str.title()

# Fix country abbreviations
df["Country"] = df["Country"].replace({
    "Uk": "UK",
    "Usa": "USA"
})

# Remove duplicate rows
df = df.drop_duplicates()

# Create useful dashboard columns
df["Churn_Status"] = df["Churned"].map({
    0: "Active",
    1: "Churned"
})

df["Payment_Mode"] = df["Payment_Method_Diversity"].map({
    1: "Cash",
    2: "Debit Card",
    3: "Credit Card",
    4: "PayPal",
    5: "Others"
})

df["Age_Group"] = pd.cut(
    df["Age"],
    bins=[16, 25, 35, 45, 55, 100],
    labels=["18-25", "26-35", "36-45", "46-55", "55 and above"]
)

df["Engagement_Tier"] = pd.cut(
    df["Login_Frequency"],
    bins=[-1, 4, 12, 24, 100],
    labels=["Low", "Medium", "High", "Very High"]
)

df["Value_Segment"] = pd.qcut(
    df["Lifetime_Value"],
    q=4,
    labels=["Bronze", "Silver", "Gold", "Platinum"]
)

# Save cleaned file
df.to_csv("churn_dashboard_clean.csv", index=True)